# 07 — Junior Author EDA
Exploratory plots for junior best-paper award winners.  
**Input files:**
- `junior_profiles_all.csv` — OpenAlex author profiles (one row per unique author)
- `junior_authors_all_conferences.csv` — award-level records (one row per author × award)

Falls back to `junior_profiles.csv` / `junior_authors_2000_2018.csv` if the new files aren't ready yet.

In [12]:
import pandas as pd
import numpy as np
import json, ast, os
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = 'plotly_white'

# Greyscale shades for bars/boxes
GREY = ['#111111','#444444','#777777','#999999','#bbbbbb','#dddddd','#eeeeee']

# Marker symbols for line plots (shape-based differentiation)
SYMBOLS = ['circle','square','diamond','triangle-up','cross','star','triangle-down']

BASE = dict(
    paper_bgcolor='white', plot_bgcolor='white',
    font=dict(family='Arial', size=13, color='#111'),
    title_font_size=20,
    margin=dict(l=70, r=50, t=90, b=70),
)

os.makedirs('figures', exist_ok=True)
print('Setup complete')


Setup complete


In [13]:
PROFILES = 'B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\profiles\\junior_profiles_all.csv' 
AUTHORS  = 'B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\matched\\junior_authors_all_conferences.csv' 
profiles = pd.read_csv(PROFILES)
authors  = pd.read_csv(AUTHORS)

merged = profiles.merge(
    authors[['author_id','author_position']].drop_duplicates('author_id'),
    on='author_id', how='left'
)

print(f'Using: {PROFILES}  →  {len(profiles):,} authors')
print(profiles['conference'].value_counts().to_string())


Using: B:\Semester 4 UU\thesis-best-paper-trajectories\data\profiles\junior_profiles_all.csv  →  603 authors
conference
CHI           119
ICSE           81
FSE            47
UIST           27
PLDI           26
OSDI           22
SOSP           22
ACL            19
VLDB           18
AAAI           16
WWW            16
INFOCOM        16
SIGCOMM        12
KDD            12
SIGMOD         12
PODS           12
CIKM           12
ICML           12
CVPR           11
FOCS           11
STOC           10
IJCAI          10
SIGMETRICS      9
SIGIR           9
MOBICOM         9
ICCV            8
S&P             8
NeurIPS         7
NSDI            5
SODA            5


In [14]:
def parse_cby(raw):
    if isinstance(raw, list): return raw
    if pd.isna(raw): return []
    try:   return json.loads(raw.replace("'", '"'))
    except:
        try: return ast.literal_eval(raw)
        except: return []

rel_rows = []
for _, row in profiles.iterrows():
    for e in parse_cby(row['counts_by_year']):
        yr = e.get('year')
        if yr:
            rel_rows.append({
                'rel_year'  : yr - row['award_year'],
                'works'     : e.get('works_count', 0),
                'citations' : e.get('cited_by_count', 0),
            })

rel_df = pd.DataFrame(rel_rows)
print(f'Trajectory rows: {len(rel_df):,}')


Trajectory rows: 6,464


#### Plot 1: Career age at award

In [15]:
age = profiles['career_age_at_award'].value_counts().sort_index().reset_index()
age.columns = ['career_age', 'count']
age['pct'] = (age['count'] / age['count'].sum() * 100).round(1)

fig = go.Figure(go.Bar(
    x=age['career_age'], y=age['count'],
    marker=dict(
        color='white',
        line=dict(color='black', width=1.5),
        pattern=dict(shape='/', size=8, solidity=0.4)
    ),
    text=[f"{c}  ({p}%)" for c, p in zip(age['count'], age['pct'])],
    textposition='outside', cliponaxis=False,
))
fig.update_layout(**BASE,
    title='Career age at time of best paper award',
    xaxis=dict(title='Years since first publication', tickmode='array',
               tickvals=list(range(6)), gridcolor='#ddd', linecolor='black'),
    yaxis=dict(title='# Authors', range=[0, age['count'].max()*1.25],
               gridcolor='#ddd', linecolor='black'),
)
fig.show()
fig.write_image('B:\\Semester 4 UU\\thesis-best-paper-trajectories\\data\\profiles\\figures/01_career_age.png')


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido
